In [6]:
from pathlib import Path

import gymnasium as gym
import numpy as np
from gymnasium.wrappers import RecordVideo
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.ppo import MlpPolicy

from imitation.algorithms import bc
from imitation.data import rollout
from imitation.data.wrappers import RolloutInfoWrapper

In [ ]:
def record_video(env_name, policy, file_name=None, deterministic=True, max_steps=1000, debug=True):
    """Roll a single episode and store the video under videos/<file_name>."""
    if debug:
        print(f"record_video: starting capture for {env_name} -> {file_name or 'unnamed-policy'}")

    file_name = file_name or "unnamed-policy"
    video_dir = Path("videos") / file_name
    video_dir.mkdir(parents=True, exist_ok=True)

    env = RecordVideo(
        gym.make(env_name, render_mode="rgb_array"),
        video_folder=str(video_dir),
        name_prefix=file_name.replace(" ", "_"),
        episode_trigger=lambda _: True,
    )
    obs, _ = env.reset()
    done = False
    steps = 0
    while not done and steps < max_steps:
        if hasattr(policy, "predict"):
            action, _ = policy.predict(obs, deterministic=deterministic)
        else:
            action = policy(obs)
        obs, _, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        steps += 1
    env.close()

    videos = sorted(video_dir.glob("*.mp4"))
    if videos and debug:
        print(f"record_video: saved video to {videos[-1]}")
    elif not videos and debug:
        print("record_video: no video files found")
    return videos[-1] if videos else None

def record_trajectory_video(trajectory, env_name, file_name=None, debug=True):
    """Replay a recorded trajectory and store the resulting video."""
    file_name = file_name or "trajectory"
    video_dir = Path("videos") / file_name
    video_dir.mkdir(parents=True, exist_ok=True)

    env = RecordVideo(
        gym.make(env_name, render_mode="rgb_array"),
        video_folder=str(video_dir),
        name_prefix=file_name.replace(" ", "_"),
        episode_trigger=lambda _: True,
    )

    try:
        initial_obs = getattr(trajectory, "obs", None)
        initial_obs = initial_obs[0] if initial_obs is not None else None
        env.reset()
        if initial_obs is not None and hasattr(env.unwrapped, "state"):
            try:
                if isinstance(initial_obs, dict):
                    raise TypeError('dict observations are not supported for trajectory replay')
                env.unwrapped.state = np.array(initial_obs, dtype=float).copy()
                if hasattr(env.unwrapped, "steps_beyond_terminated"):
                    env.unwrapped.steps_beyond_terminated = None
            except Exception as exc:
                if debug:
                    print(f"record_trajectory_video: failed to set env state: {exc}")

        for action in getattr(trajectory, "acts", []):
            step_action = action
            if isinstance(step_action, np.ndarray):
                if step_action.shape == ():
                    step_action = step_action.item()
                else:
                    step_action = step_action.astype(float)
            _, _, terminated, truncated, _ = env.step(step_action)
            if terminated or truncated:
                break
    finally:
        env.close()

    videos = sorted(video_dir.glob("*.mp4"))
    if videos and debug:
        print(f"record_trajectory_video: saved video to {videos[-1]}")
    elif not videos and debug:
        print("record_trajectory_video: no video files found")
    return videos[-1] if videos else None


def train_and_save_policy(iterations, env_name, file_name=None, policy=None, debug=True):
    """Train the supplied policy (or a fresh PPO policy) and persist it to disk."""
    file_name = file_name or "unnamed-policy"

    if debug:
        print(f"train_and_save_policy: training for {iterations} timesteps on {env_name} -> {file_name}")

    if policy is None:
        env = gym.make(env_name)
        model = PPO(
            policy=MlpPolicy,
            env=env,
            seed=0,
            batch_size=64,
            ent_coef=0.0,
            learning_rate=0.0003,
            n_epochs=10,
            n_steps=64,
        )
        env_to_close = env
    else:
        model = policy
        env_to_close = None
        if hasattr(model, "get_env"):
            current_env = model.get_env()
            if current_env is None:
                env = gym.make(env_name)
                model.set_env(env)
                env_to_close = env
        else:
            raise ValueError("Provided policy must expose get_env() and set_env().")

    model.learn(total_timesteps=iterations)

    save_dir = Path("policies")
    save_dir.mkdir(parents=True, exist_ok=True)
    save_path = save_dir / file_name
    model.save(str(save_path))

    if debug:
        print(f"train_and_save_policy: saved policy to {save_path}")

    if env_to_close is not None:
        env_to_close.close()

    return model


In [8]:
ENV_NAME = "CartPole-v1"
ITERATION_STEP = 1000
NUM_POLICIES = 10

policy = None
video_entries = []
policy_entries = []

for idx in range(1, NUM_POLICIES + 1):
    total_timesteps = idx * ITERATION_STEP
    policy_filename = f"ppo_cartpole_{total_timesteps}"
    policy_path = Path("policies") / f"{policy_filename}.zip"
    video_dir = Path("videos") / policy_filename
    existing_videos = sorted(video_dir.glob("*.mp4")) if video_dir.exists() else []
    policy_entries.append(policy_path)
    if policy_path.exists() and existing_videos:
        print(f"{policy_filename} already exists.")
        policy = PPO.load(str(policy_path))
        video_entries.append((total_timesteps, existing_videos[-1]))
        continue

    policy = train_and_save_policy(
        iterations=ITERATION_STEP,
        env_name=ENV_NAME,
        file_name=policy_filename,
        policy=policy,
        debug=True,
    )
    video_path = record_video(
        env_name=ENV_NAME,
        policy=policy,
        file_name=policy_filename,
        debug=True,
    )
    video_entries.append((total_timesteps, video_path))

ppo_cartpole_1000 already exists.
ppo_cartpole_2000 already exists.
ppo_cartpole_3000 already exists.
ppo_cartpole_4000 already exists.
ppo_cartpole_5000 already exists.
ppo_cartpole_6000 already exists.
ppo_cartpole_7000 already exists.
ppo_cartpole_8000 already exists.
ppo_cartpole_9000 already exists.
ppo_cartpole_10000 already exists.


In [9]:
from IPython.display import HTML

html_parts = []
for timesteps, path in video_entries:
    if path is None:
        continue
    rel_path = path.as_posix()
    html_parts.append(
        f'<div class="video-item"><h4>{timesteps:,} steps - policy</h4>'
        f'<video controls loop muted src="{rel_path}"></video></div>'
    )

items_html = "".join(html_parts)

grid_html = f"""
<style>
.video-grid {{display:flex; flex-wrap:wrap; gap:16px;}}
.video-grid .video-item {{width: 240px;}}
.video-grid h4 {{margin: 0 0 8px 0; font-size: 14px;}}
.video-grid video {{width: 100%; height: auto; border:1px solid #ccc; border-radius:4px;}}
</style>
<div class="video-grid">
{items_html}
</div>
"""

HTML(grid_html)


In [10]:
def sample_trajectories_with_reward(policy, no_traj, env_name=ENV_NAME, seed=None, deterministic=True):
    """Collect `no_traj` trajectories and their returns from the given policy."""
    if no_traj <= 0:
        return []

    if hasattr(policy, "get_env") and policy.get_env() is not None:
        vec_env = policy.get_env()
        close_env = False
    else:
        vec_env = DummyVecEnv([lambda: RolloutInfoWrapper(gym.make(env_name))])
        close_env = True

    try:
        sample_until = rollout.make_sample_until(min_timesteps=None, min_episodes=no_traj)
        trajectories = rollout.rollout(
            policy,
            vec_env,
            sample_until,
            rng=np.random.default_rng(seed),
            deterministic_policy=deterministic,
        )
    finally:
        if close_env:
            vec_env.close()

    results = []
    for traj in trajectories[:no_traj]:
        total_reward = float(np.sum(traj.rews))
        results.append({"trajectory": traj, "reward": total_reward})

    return results


In [43]:
trajectory_entries = []

for policy_path in policy_entries:
    if not policy_path.exists():
        continue
    policy = PPO.load(str(policy_path))

    traj_infos = sample_trajectories_with_reward(
        policy=policy,
        no_traj=3,
        env_name=ENV_NAME,
        deterministic=True,
    )
    
    trajectory_entries.extend(traj_infos)



best_entry = max(trajectory_entries, key=lambda entry: entry["reward"])
worst_entry = min(trajectory_entries, key=lambda entry: entry["reward"])

best_video = record_trajectory_video(best_entry["trajectory"], ENV_NAME, "trajectory_best")
worst_video = record_trajectory_video(worst_entry["trajectory"], ENV_NAME, "trajectory_worst")


/home/mzoellner/Projects/purdue/imitation-learning/imitation/.pixi/envs/default/lib/python3.10/site-packages/gymnasium/wrappers/record_video.py:94: UserWarning: WARN: Overwriting existing videos at /home/mzoellner/Projects/purdue/imitation-learning/imitation/tests/videos/trajectory_best folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


MoviePy - Building video /home/mzoellner/Projects/purdue/imitation-learning/imitation/tests/videos/trajectory_best/trajectory_best-episode-0.mp4.
MoviePy - Writing video /home/mzoellner/Projects/purdue/imitation-learning/imitation/tests/videos/trajectory_best/trajectory_best-episode-0.mp4



/home/mzoellner/Projects/purdue/imitation-learning/imitation/.pixi/envs/default/lib/python3.10/site-packages/gymnasium/wrappers/record_video.py:94: UserWarning: WARN: Overwriting existing videos at /home/mzoellner/Projects/purdue/imitation-learning/imitation/tests/videos/trajectory_worst folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


MoviePy - Done !
MoviePy - video ready /home/mzoellner/Projects/purdue/imitation-learning/imitation/tests/videos/trajectory_best/trajectory_best-episode-0.mp4
record_trajectory_video: saved video to videos/trajectory_best/trajectory_best-episode-0.mp4
MoviePy - Building video /home/mzoellner/Projects/purdue/imitation-learning/imitation/tests/videos/trajectory_worst/trajectory_worst-episode-0.mp4.
MoviePy - Writing video /home/mzoellner/Projects/purdue/imitation-learning/imitation/tests/videos/trajectory_worst/trajectory_worst-episode-0.mp4



MoviePy - Done !
MoviePy - video ready /home/mzoellner/Projects/purdue/imitation-learning/imitation/tests/videos/trajectory_worst/trajectory_worst-episode-0.mp4
record_trajectory_video: saved video to videos/trajectory_worst/trajectory_worst-episode-0.mp4


In [ ]:
display_items = []
if worst_video is not None:
    label = "Worst trajectory"
    label += f": reward {worst_entry['reward']:.2f}"
    display_items.append((label, worst_video))

if best_video is not None:
    label = "Best trajectory"
    label += f": reward {best_entry['reward']:.2f}"
    display_items.append((label, best_video))

if not display_items:
    print("No videos available for best/worst trajectories.")
else:
    html_parts = []
    for label, path in display_items:
        rel_path = path.as_posix() if hasattr(path, "as_posix") else str(path)
        html_parts.append(
            f'<div class="video-item"><h4>{label}</h4>'
            f'<video controls loop muted src="{rel_path}"></video></div>'
        )

    grid_html = f"""
<style>
.video-grid {{display:flex; flex-wrap:wrap; gap:16px;}}
.video-grid .video-item {{width: 320px;}}
.video-grid h4 {{margin: 0 0 8px 0; font-size: 14px;}}
.video-grid video {{width: 100%; height: auto; border:1px solid #ccc; border-radius:4px;}}
</style>
<div class="video-grid">
{''.join(html_parts)}
</div>
"""

HTML(grid_html)

In [ ]:
for i in range(0, len(trajectory_entries)):
    print(trajectory_entries[i]["reward"]) # why is reward capped at 500 ? 

60.0
42.0
95.0
134.0
122.0
145.0
267.0
304.0
231.0
500.0
500.0
500.0
500.0
186.0
325.0
500.0
500.0
500.0
129.0
126.0
143.0
496.0
500.0
500.0
500.0
500.0
500.0
500.0
500.0
500.0


In [ ]:
# get pairs, setup neural network, train reward loss function 